In [0]:
path='development_042_silver_sandbox.demand_forecast.sales_bronz_dup'
mpath='development_042_silver_sandbox.demand_forecast.eilers_to_gdna_mapping_silver'
dest='development_042_silver_sandbox.demand_forecast.sales_silver'

In [0]:
df=spark.table(path)
mdf=spark.table(mpath)
display(mdf)
display(df)

In [0]:
import re
from pyspark.sql.functions import col, lower, regexp_replace

def clean_col_name(name):
    # Lowercase and remove non-alphanumeric/underscore
    return re.sub(r'[^a-z0-9_]', '', name.lower())

# Rename columns removing uncessary characters and droping spaces
df_clean = df
for c in df_clean.columns:
    df_clean = df_clean.withColumnRenamed(c, clean_col_name(c))

# Lowercase all string columns and remove bad characters
for c, dtype in df_clean.dtypes:
    if dtype == 'string':
        df_clean = df_clean.withColumn(
            c,
            regexp_replace(lower(col(c)), r'[^a-z0-9_ ]', '')
        )

display(df_clean)

In [0]:
df_filtered = df_clean.filter(col("revenue_type").isin(["sale", "lease", "rental"])) # only keeping renatls and sold units
df_filtered = df_filtered.filter(col("ref_num_region").isin(["east", "west", "canada"])) # the only markets we care about are US east, west and Canada
display(df_filtered)

In [0]:
from pyspark.sql import functions as F

df_grouped = df_filtered.groupBy("friendly_game_name", "beginning_month_date", "revenue_type") \
    .agg(F.sum("expr1").alias("expr1_sum"))
display(df_grouped)

In [0]:
import re
import unicodedata
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

DROP_TOKENS = {
    "asc", "cc", "mlp", "cds", "dnu", "c2", "sap",
    "cd", "crv", "australia", "cmb", "uk", "bc",
    "dl", "drs", "fg", "hhr", "pt", "sa", "intl",
    "lc", "po", "portrait", "us", "video", "avp",
    "bap", "gk",
    "3r", "5r", "3d", "4d",
    "3r1l", "3r1l2c", "3r1l3c",
    "3r5l", "3r5l10c", "3r9l", "3r15l",
    "3r20l", "3r25l", "3r27l",
    "5r15l", "5r20l", "5r25l", "5r30l",
    "5r40l", "5r50l", "5r75l", "6r50l",
    "30l", "50l", "60ctc", "90c"
}

def normalize_sales_title(name: str) -> str:
    name = unicodedata.normalize("NFKD", str(name))
    name = name.encode("ascii", "ignore").decode()
    name = name.lower()
    name = re.sub(r"[^a-z0-9]+", " ", name).strip()
    tokens = []
    for token in name.split():
        if token == "wof":
            tokens.extend(["wheel", "of", "fortune"])
        else:
            tokens.append(token)
    tokens = [token for token in tokens if token not in DROP_TOKENS]
    while tokens and tokens[-1] == "dual":
        tokens.pop()
    return " ".join(tokens)

normalize_sales_title_udf = udf(normalize_sales_title, StringType())

df_grouped = df_grouped.withColumn(
    "normalized_game_name",
    normalize_sales_title_udf(col("friendly_game_name"))
)
display(df_grouped)

In [0]:
df_joined = df_grouped.join(
    mdf,
    df_grouped.normalized_game_name== mdf['sales_normalized_title'],
    how="left"
)
display(df_joined)

In [0]:
df_final = df_joined.filter(
    (col("sales_match_score").cast("double") >= 80) & (~col("correction").eqNullSafe("no_match"))
).drop("correction", "bp_theme_name", "friendly_game_name")
display(df_final)

In [0]:
df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(dest)